In [ ]:
import re
import time
import json
from pathlib import Path

import pandas as pd
from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC


# =========================
# 설정
# =========================
TARGET_N = 1000
SAVE_EVERY = 50

BASE_SAVE_DIR = Path("encar_chevrolet_all")
HTML_DIR = BASE_SAVE_DIR / "html"
OUTPUT_DIR = BASE_SAVE_DIR / "output"

BATCH_CSV = OUTPUT_DIR / "chevrolet_detail_batch.csv"
ERROR_CSV = OUTPUT_DIR / "chevrolet_detail_errors.csv"
FINAL_JSON = OUTPUT_DIR / "chevrolet_detail_final.json"

SEARCH_URL = (
    "https://car.encar.com/list/car?page=1&search=%7B%22type%22%3A%22car%22%2C"
    "%22action%22%3A%22(And.Hidden.N._.(C.CarType.Y._.Manufacturer.%EC%89%90%EB%B3%B4%EB%A0%88(GM%EB%8C%80%EC%9A%B0_).))%22%2C"
    "%22title%22%3A%22%EC%89%90%EB%B3%B4%EB%A0%88(GM%EB%8C%80%EC%9A%B0)%22%2C"
    "%22toggle%22%3A%7B%7D%2C%22layer%22%3A%22%22%2C%22sort%22%3A%22MobileModifiedDate%22%7D"
)


# =========================
# 공통 유틸
# =========================
def normalize_text(text):
    return re.sub(r"\s+", " ", str(text)).strip() if text else ""


def find_first(patterns, text, cast=None, flags=re.S):
    for pat in patterns:
        m = re.search(pat, text, flags)
        if m:
            val = m.group(1) if m.lastindex else m.group(0)
            if cast:
                try:
                    return cast(val)
                except Exception:
                    return val
            return val
    return None


def to_int(val):
    if val is None:
        return None
    return int(str(val).replace(",", "").strip())


def parse_int_from_text(text):
    if not text:
        return None
    m = re.search(r"([\d,]+)", text)
    return int(m.group(1).replace(",", "")) if m else None


def parse_json_array_str(text):
    if not text:
        return []
    try:
        return json.loads(text)
    except Exception:
        return []


def extract_car_id(url):
    m = re.search(r"/cars/detail/(\d+)", url)
    return m.group(1) if m else None


def build_full_trim(model, grade, detail):
    parts = []
    for p in [model, grade, detail]:
        p = normalize_text(p)
        if p and p not in parts:
            parts.append(p)
    return " ".join(parts) if parts else None


def convert_year_from_embedded(year_month, form_year, fallback_text):
    if form_year:
        try:
            return int(form_year)
        except Exception:
            pass
    if year_month and len(year_month) >= 4:
        try:
            return int(year_month[:4])
        except Exception:
            pass
    if fallback_text:
        m = re.search(r"(\d{2})/", fallback_text)
        if m:
            return 2000 + int(m.group(1))
    return None


# =========================
# Selenium
# =========================
def setup_driver(headless=False):
    options = Options()
    if headless:
        options.add_argument("--headless=new")
    options.add_argument("--start-maximized")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument("--lang=ko-KR")
    options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"
    )
    driver = webdriver.Chrome(options=options)
    driver.implicitly_wait(3)
    return driver


def wait_body(driver, sec=10):
    WebDriverWait(driver, sec).until(
        EC.presence_of_element_located((By.TAG_NAME, "body"))
    )


def close_popups(driver):
    popup_xpaths = [
        "//button[contains(., '닫기')]",
        "//button[contains(., '나중에')]",
        "//button[contains(., '오늘 보지 않기')]",
        "//button[contains(., '취소')]",
        "//button[contains(., '확인')]",
        "//button[contains(., '다음에')]",
        "//a[contains(., '닫기')]",
    ]
    for xp in popup_xpaths:
        try:
            buttons = driver.find_elements(By.XPATH, xp)
            for btn in buttons:
                try:
                    if btn.is_displayed():
                        driver.execute_script("arguments[0].click();", btn)
                        time.sleep(0.2)
                except Exception:
                    pass
        except Exception:
            pass


def collect_detail_links(driver):
    anchors = driver.find_elements(By.CSS_SELECTOR, 'a[href*="/cars/detail/"]')
    urls = []
    for a in anchors:
        try:
            href = a.get_attribute("href")
            if href and "/cars/detail/" in href:
                href = href.split("&advClickPosition=")[0]
                urls.append(href)
        except Exception:
            pass
    return list(dict.fromkeys(urls))


def collect_chevrolet_links(driver, target_n=1000, max_scrolls=300):
    driver.get(SEARCH_URL)
    wait_body(driver)
    time.sleep(2)
    close_popups(driver)

    links = []
    last_count = 0
    stable_round = 0

    for i in range(max_scrolls):
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(2)

        links = collect_detail_links(driver)
        current_count = len(links)
        print(f"[SCROLL {i+1}] 링크 수: {current_count}")

        if current_count >= target_n:
            return links[:target_n]

        if current_count == last_count:
            stable_round += 1
        else:
            stable_round = 0

        if stable_round >= 5:
            print("[INFO] 더 이상 링크가 늘지 않아 중단")
            break

        last_count = current_count

    return links[:target_n]


def fetch_and_save_detail_page(driver, url, html_path, txt_path=None, sleep_sec=2.0):
    driver.get(url)
    wait_body(driver)
    time.sleep(sleep_sec)
    close_popups(driver)
    time.sleep(0.5)

    html = driver.page_source
    html_path.write_text(html, encoding="utf-8")

    if txt_path is not None:
        body_text = normalize_text(driver.find_element(By.TAG_NAME, "body").text)
        txt_path.write_text(body_text, encoding="utf-8")

    return html


# =========================
# detail 파싱
# =========================
def parse_meta_description(soup):
    result = {}
    meta = soup.find("meta", attrs={"name": "description"})
    if not meta:
        return result

    content = meta.get("content", "")
    result["연식_메타"] = find_first([r"연식:([^,]+)"], content)
    result["주행거리_메타"] = find_first([r"주행거리:([^,]+)"], content)
    result["연료_메타"] = find_first([r"연료:([^,]+)"], content)
    result["색상_메타"] = find_first([r"색상:([^,]+)"], content)
    result["지역_메타"] = find_first([r"지역:([^,]+?) 중고차"], content)
    return result


def parse_dom_basic(soup):
    result = {
        "차량명_dom": None,
        "모델_dom": None,
        "세부트림_dom": None,
        "연식_원문_dom": None,
        "주행거리_원문_dom": None,
        "연료_dom": None,
        "차량번호_dom": None,
        "등록번호_dom": None,
        "조회수_dom": None,
        "찜수_dom": None,
        "해시태그_dom": None,
        "실촬영문구_dom": None,
    }

    h3 = soup.find("h3")
    if h3:
        spans = [normalize_text(x.get_text(" ", strip=True)) for x in h3.find_all("span")]
        spans = [x for x in spans if x]
        if len(spans) >= 2:
            result["모델_dom"] = spans[0]
            result["세부트림_dom"] = spans[1]
            result["차량명_dom"] = " ".join(spans).strip()
        else:
            result["차량명_dom"] = normalize_text(h3.get_text(" ", strip=True))

    dl = soup.select_one("dl.ar1Ivd7EgX")
    if dl:
        dts = dl.find_all("dt")
        dds = dl.find_all("dd")
        for dt, dd in zip(dts, dds):
            k = normalize_text(dt.get_text(" ", strip=True))
            v = normalize_text(dd.get_text(" ", strip=True))
            if "연식" in k:
                result["연식_원문_dom"] = v
            elif "주행거리" in k:
                result["주행거리_원문_dom"] = v
            elif "연료" in k:
                result["연료_dom"] = v
            elif "차량번호" in k:
                result["차량번호_dom"] = v

    for li in soup.select("ul.rVigc5A_1H li"):
        txt = normalize_text(li.get_text(" ", strip=True))
        if txt.startswith("등록번호"):
            result["등록번호_dom"] = txt.replace("등록번호", "").strip()
        elif txt.startswith("조회수"):
            result["조회수_dom"] = parse_int_from_text(txt)
        elif txt.startswith("찜"):
            result["찜수_dom"] = parse_int_from_text(txt)

    tags = [normalize_text(li.get_text(" ", strip=True)) for li in soup.select("ul.kYC8KS_YZ1 li")]
    tags = [x for x in tags if x]
    result["해시태그_dom"] = ",".join(tags) if tags else None

    shot = soup.select_one("p.s8FzQTBVpE")
    if shot:
        result["실촬영문구_dom"] = normalize_text(shot.get_text(" ", strip=True))

    return result


def parse_detail_embedded(html):
    result = {}

    str_patterns = {
        "manufacturerName": [r'"manufacturerName":"([^"]+)"'],
        "modelName": [r'"modelName":"([^"]+)"'],
        "gradeName": [r'"gradeName":"([^"]+)"'],
        "gradeDetailName": [r'"gradeDetailName":"([^"]+)"'],
        "vehicleNo": [r'"vehicleNo":"([^"]+)"'],
        "vin": [r'"vin":"([^"]+)"'],
        "requestUrl": [r'"requestUrl":"([^"]+)"'],
        "dealerName": [r'"dealer":\{"userId":"[^"]+","name":"([^"]+)"'],
        "firmName": [r'"firm":\{"code":"[^"]+","name":"([^"]+)"'],
        "diagnosisCenterName": [r'"diagnosisCenters":\[\{"code":"[^"]+","name":"([^"]+)"'],
        "diagnosisCenterPhone": [r'"telephoneNumber":"([^"]+)"'],
        "diagnosisCenterAddress": [r'"address":"([^"]+)"'],
        "pageAccessToken": [r'"pageAccessToken":"([^"]+)"'],
        "yearMonth": [r'"yearMonth":"([^"]+)"'],
        "formYear": [r'"formYear":"([^"]+)"'],
        "fuelName": [r'"fuelName":"([^"]+)"'],
        "colorName": [r'"colorName":"([^"]+)"'],
        "transmissionName": [r'"transmissionName":"([^"]+)"'],
        "bodyName": [r'"bodyName":"([^"]+)"'],
        "oneLineText": [r'"oneLineText":"([^"]+)"'],
    }

    int_patterns = {
        "price": [r'"price":(\d+)'],
        "viewCount": [r'"viewCount":(\d+)'],
        "subscribeCount": [r'"subscribeCount":(\d+)'],
        "vehicleId": [r'"vehicleId":(\d+)'],
        "seizingCount": [r'"seizingCount":(\d+)'],
        "pledgeCount": [r'"pledgeCount":(\d+)'],
        "encarDiagnosis": [r'"encarDiagnosis":(-?\d+)'],
        "encarMeetGo": [r'"encarMeetGo":(-?\d+)'],
        "mileage": [r'"mileage":(\d+)'],
        "displacement": [r'"displacement":(\d+)'],
        "seatCount": [r'"seatCount":(\d+)'],
    }

    for key, pats in str_patterns.items():
        result[key] = find_first(pats, html)

    for key, pats in int_patterns.items():
        result[key] = find_first(pats, html, cast=to_int)

    standard_raw = find_first([r'"standard":(\[[^\]]*\])'], html)
    choice_raw = find_first([r'"choice":(\[[^\]]*\])'], html)
    tuning_raw = find_first([r'"tuning":(\[[^\]]*\])'], html)
    etc_raw = find_first([r'"etc":(\[[^\]]*\]|"[^"]*")'], html)

    result["option_standard_codes"] = parse_json_array_str(standard_raw)
    result["option_choice_codes"] = parse_json_array_str(choice_raw)
    result["option_tuning_codes"] = parse_json_array_str(tuning_raw)

    if etc_raw and etc_raw.startswith("["):
        result["option_etc_values"] = parse_json_array_str(etc_raw)
    elif etc_raw:
        result["option_etc_values"] = [etc_raw.strip('"')]
    else:
        result["option_etc_values"] = []

    return result


def parse_dom_major_options(soup):
    result = {}
    for li in soup.select("ul.dz3qFYruNO li"):
        text = normalize_text(li.get_text(" ", strip=True))
        blind = li.select_one("span.blind")
        status = normalize_text(blind.get_text(" ", strip=True)) if blind else None

        if status:
            name = normalize_text(text.replace(status, ""))
            result[f"주요옵션_{name}"] = 1 if status == "있음" else 0 if status == "없음" else None

    return result


def parse_dom_seller_info(soup):
    result = {
        "판매자상호_dom": None,
        "판매자명_dom": None,
        "판매자유형_dom": None,
        "판매중대수_dom": None,
        "판매완료대수_dom": None,
        "판매자지역_dom": None,
        "종사원증번호_dom": None,
    }

    btn = soup.select_one("button.uPEfVsKnZx")
    if btn:
        brand = btn.select_one("span.ESlvHTih77")
        name = btn.select_one("strong.k3tS4rXdrQ")
        seller_type = btn.select_one("span.xf9ufEfgKR")

        if brand:
            result["판매자상호_dom"] = normalize_text(brand.get_text(" ", strip=True))
        if name:
            result["판매자명_dom"] = normalize_text(name.get_text(" ", strip=True))
        if seller_type:
            result["판매자유형_dom"] = normalize_text(seller_type.get_text(" ", strip=True))

    lis = soup.select("ul.VtNR8dNHOS li")
    for li in lis:
        txt = normalize_text(li.get_text(" ", strip=True))
        if txt.startswith("판매중"):
            result["판매중대수_dom"] = parse_int_from_text(txt)
        elif txt.startswith("판매완료"):
            result["판매완료대수_dom"] = parse_int_from_text(txt)
        elif "종사원증번호" in txt:
            m = re.search(r"종사원증번호\s*([A-Z0-9\-]+)", txt)
            if m:
                result["종사원증번호_dom"] = m.group(1)
        elif any(region in txt for region in ["서울", "경기", "인천", "부산", "대구", "대전", "광주", "울산", "세종", "강원", "충북", "충남", "전북", "전남", "경북", "경남", "제주"]):
            result["판매자지역_dom"] = txt

    return result


def parse_dom_extra_flags(soup):
    result = {
        "보험이력공개여부_dom": None,
        "보험이력건수_dom": None,
        "성능점검유형_dom": None,
        "성능점검설명_dom": None,
    }

    for btn in soup.select("button.fSZoGuAX4M"):
        txt = normalize_text(btn.get_text(" ", strip=True))
        if "보험이력" in txt:
            result["보험이력공개여부_dom"] = txt
            m = re.search(r"보험이력\s*(\d+)건", txt)
            if m:
                result["보험이력건수_dom"] = int(m.group(1))
        elif "성능점검내역" in txt:
            result["성능점검유형_dom"] = txt.replace("성능점검내역", "").strip()

    perf_desc = soup.select_one("div.Hs3feBPsTj p.lGhsYmQaGE")
    if perf_desc:
        result["성능점검설명_dom"] = normalize_text(perf_desc.get_text(" ", strip=True))

    return result


def parse_damage_detail(soup):
    result = {
        "교환_개수": None,
        "판금_개수": None,
        "부식_여부": None,
        "성능기록부_raw": None,
    }

    items = []
    for li in soup.select("ul.wB0X7nC0cq li"):
        txt = normalize_text(li.get_text(" ", strip=True))
        if txt:
            items.append(txt)

        if "교환" in txt:
            if "없음" in txt:
                result["교환_개수"] = 0
            else:
                m = re.search(r"교환\s*(\d+)", txt)
                if m:
                    result["교환_개수"] = int(m.group(1))

        elif "판금" in txt:
            if "없음" in txt:
                result["판금_개수"] = 0
            else:
                m = re.search(r"판금\s*(\d+)", txt)
                if m:
                    result["판금_개수"] = int(m.group(1))

        elif "부식" in txt:
            result["부식_여부"] = 0 if "없음" in txt else 1

    result["성능기록부_raw"] = " || ".join(items) if items else None
    return result


def derive_accident_features(row):
    exchange_cnt = row.get("교환_개수")
    panel_cnt = row.get("판금_개수")
    corrosion = row.get("부식_여부")
    insurance_cnt = row.get("보험이력건수")

    row["교환_여부"] = None if exchange_cnt is None else int(exchange_cnt > 0)
    row["판금_여부"] = None if panel_cnt is None else int(panel_cnt > 0)
    row["외판수리_여부"] = None if (exchange_cnt is None and panel_cnt is None) else int((exchange_cnt or 0) + (panel_cnt or 0) > 0)

    if exchange_cnt is None and panel_cnt is None:
        row["사고강도점수"] = None
    else:
        row["사고강도점수"] = (exchange_cnt or 0) * 2 + (panel_cnt or 0)

    row["보험이력_여부"] = None if insurance_cnt is None else int(insurance_cnt > 0)

    signals = []
    for v in [row["교환_여부"], row["판금_여부"], row["보험이력_여부"]]:
        if v is not None:
            signals.append(v)

    row["사고종합_여부"] = None if not signals else int(any(signals))

    if row["사고강도점수"] is None:
        row["중대사고_추정"] = None
    else:
        row["중대사고_추정"] = int((exchange_cnt or 0) >= 2 or row["사고강도점수"] >= 4)

    row["부식_위험"] = corrosion
    return row


def parse_detail_file(detail_html: str, car_id_hint=None):
    soup = BeautifulSoup(detail_html, "lxml")

    meta = parse_meta_description(soup)
    dom = parse_dom_basic(soup)
    emb = parse_detail_embedded(detail_html)
    major_options = parse_dom_major_options(soup)
    seller_dom = parse_dom_seller_info(soup)
    flags_dom = parse_dom_extra_flags(soup)
    damage_dom = parse_damage_detail(soup)

    model = emb.get("modelName") or dom.get("모델_dom")
    grade = emb.get("gradeName")
    detail = emb.get("gradeDetailName") or dom.get("세부트림_dom")

    if normalize_text(grade) == normalize_text(detail):
        detail = None

    row = {
        "매물ID": emb.get("vehicleId") or car_id_hint,
        "제조사": emb.get("manufacturerName"),
        "모델": model,
        "등급명": grade,
        "세부트림": detail,
        "차량명": build_full_trim(model, grade, detail),
        "현재가격_만원": emb.get("price"),
        "조회수": emb.get("viewCount") or dom.get("조회수_dom"),
        "찜수": emb.get("subscribeCount") or dom.get("찜수_dom"),
        "차량번호": emb.get("vehicleNo") or dom.get("차량번호_dom"),
        "VIN": emb.get("vin"),
        "연식_원문": dom.get("연식_원문_dom") or meta.get("연식_메타"),
        "연식": convert_year_from_embedded(emb.get("yearMonth"), emb.get("formYear"), dom.get("연식_원문_dom") or meta.get("연식_메타")),
        "주행거리_원문": dom.get("주행거리_원문_dom") or meta.get("주행거리_메타"),
        "주행거리_km": emb.get("mileage") or parse_int_from_text(dom.get("주행거리_원문_dom") or meta.get("주행거리_메타")),
        "연료": emb.get("fuelName") or dom.get("연료_dom") or meta.get("연료_메타"),
        "색상": emb.get("colorName") or meta.get("색상_메타"),
        "지역": meta.get("지역_메타"),
        "변속기": emb.get("transmissionName"),
        "배기량_cc": emb.get("displacement"),
        "차급": emb.get("bodyName"),
        "좌석수": emb.get("seatCount"),
        "등록번호": dom.get("등록번호_dom"),
        "해시태그": dom.get("해시태그_dom"),
        "실촬영문구": dom.get("실촬영문구_dom"),
        "딜러명": emb.get("dealerName") or seller_dom.get("판매자명_dom"),
        "상사명": emb.get("firmName") or seller_dom.get("판매자상호_dom"),
        "판매자유형": seller_dom.get("판매자유형_dom"),
        "판매중대수": seller_dom.get("판매중대수_dom"),
        "판매완료대수": seller_dom.get("판매완료대수_dom"),
        "판매자지역": seller_dom.get("판매자지역_dom"),
        "종사원증번호": seller_dom.get("종사원증번호_dom"),
        "진단센터명": emb.get("diagnosisCenterName"),
        "진단센터전화": emb.get("diagnosisCenterPhone"),
        "진단센터주소": emb.get("diagnosisCenterAddress"),
        "압류건수": emb.get("seizingCount"),
        "저당건수": emb.get("pledgeCount"),
        "엔카진단여부값": emb.get("encarDiagnosis"),
        "엔카믿고값": emb.get("encarMeetGo"),
        "광고한줄문구": emb.get("oneLineText"),
        "보험이력공개여부": flags_dom.get("보험이력공개여부_dom"),
        "보험이력건수": flags_dom.get("보험이력건수_dom"),
        "성능점검유형": flags_dom.get("성능점검유형_dom"),
        "성능점검설명": flags_dom.get("성능점검설명_dom"),
        "requestUrl": emb.get("requestUrl"),
        "pageAccessToken": emb.get("pageAccessToken"),
        "옵션_기본코드개수": len(emb.get("option_standard_codes", [])),
        "옵션_선택코드개수": len(emb.get("option_choice_codes", [])),
        "옵션_튜닝코드개수": len(emb.get("option_tuning_codes", [])),
        "옵션_기타개수": len(emb.get("option_etc_values", [])),
        "옵션_기본코드": ",".join(emb.get("option_standard_codes", [])) if emb.get("option_standard_codes") else None,
        "옵션_선택코드": ",".join(emb.get("option_choice_codes", [])) if emb.get("option_choice_codes") else None,
        "옵션_튜닝코드": ",".join(emb.get("option_tuning_codes", [])) if emb.get("option_tuning_codes") else None,
        "옵션_기타값": " | ".join(emb.get("option_etc_values", [])) if emb.get("option_etc_values") else None,
    }

    row.update(major_options)
    row.update(damage_dom)
    row = derive_accident_features(row)

    return row


# -------------------------
# 저장 헬퍼
# -------------------------
def save_batch(rows):
    if not rows:
        return

    df = pd.DataFrame(rows)
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    df.to_csv(BATCH_CSV, index=False, encoding="utf-8-sig")


def save_errors(errors):
    if not errors:
        return

    df = pd.DataFrame(errors)
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    df.to_csv(ERROR_CSV, index=False, encoding="utf-8-sig")


# -------------------------
# 메인
# -------------------------
def main():
    HTML_DIR.mkdir(parents=True, exist_ok=True)
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    driver = setup_driver(headless=False)
    rows = []
    errors = []

    try:
        detail_links = collect_chevrolet_links(driver, target_n=TARGET_N)
        print(f"[INFO] 수집 대상 상세링크 수: {len(detail_links)}")

        seen_ids = set()

        for idx, detail_url in enumerate(detail_links, start=1):
            car_id = extract_car_id(detail_url)

            if not car_id:
                errors.append({
                    "idx": idx,
                    "상세링크": detail_url,
                    "error": "매물ID 추출 실패"
                })
                continue

            if car_id in seen_ids:
                continue
            seen_ids.add(car_id)

            print(f"\n[{idx}/{len(detail_links)}] 매물ID={car_id}")

            try:
                car_dir = HTML_DIR / str(car_id)
                car_dir.mkdir(parents=True, exist_ok=True)

                detail_html = fetch_and_save_detail_page(
                    driver,
                    detail_url,
                    car_dir / "detail.html",
                    car_dir / "detail.txt"
                )

                row = parse_detail_file(detail_html, car_id_hint=car_id)
                row["상세링크"] = detail_url
                rows.append(row)

                print(f"[OK] {car_id} 완료 / 누적 {len(rows)}건")

                if len(rows) % SAVE_EVERY == 0:
                    print(f"[SAVE] {len(rows)}건 batch 저장")
                    save_batch(rows)
                    save_errors(errors)

                if len(rows) >= TARGET_N:
                    break

            except Exception as e:
                print(f"[ERROR] {car_id}: {e}")
                errors.append({
                    "idx": idx,
                    "매물ID": car_id,
                    "상세링크": detail_url,
                    "error": str(e)
                })

    finally:
        driver.quit()

    save_batch(rows)
    save_errors(errors)

    with open(FINAL_JSON, "w", encoding="utf-8") as f:
        json.dump(rows, f, ensure_ascii=False, indent=2)

    print(f"\n저장 완료: {BATCH_CSV}")
    print(f"저장 완료: {ERROR_CSV}")
    print(f"저장 완료: {FINAL_JSON}")
    print(f"HTML 저장 폴더: {HTML_DIR.resolve()}")
    print(f"\n최종 수집 건수: {len(rows)}")
    print(f"에러 건수: {len(errors)}")

    if rows:
        df = pd.DataFrame(rows)
        show_cols = [
            "매물ID", "차량명", "현재가격_만원",
            "교환_개수", "판금_개수", "부식_여부",
            "보험이력건수", "사고강도점수", "사고종합_여부"
        ]
        show_cols = [c for c in show_cols if c in df.columns]
        print("\n[HEAD]")
        print(df[show_cols].head())


if __name__ == "__main__":
    main()

[SCROLL 1] 링크 수: 250
[SCROLL 2] 링크 수: 250
[SCROLL 3] 링크 수: 250
[SCROLL 4] 링크 수: 250
[SCROLL 5] 링크 수: 250
[SCROLL 6] 링크 수: 250
[INFO] 더 이상 링크가 늘지 않아 중단
[INFO] 수집 대상 상세링크 수: 250

[1/250] 매물ID=40726314
[OK] 40726314 완료 / 누적 1건

[2/250] 매물ID=41275100
[OK] 41275100 완료 / 누적 2건

[3/250] 매물ID=41407659
[OK] 41407659 완료 / 누적 3건

[4/250] 매물ID=41430846
[OK] 41430846 완료 / 누적 4건

[5/250] 매물ID=40751414
[OK] 40751414 완료 / 누적 5건

[6/250] 매물ID=41314236
[OK] 41314236 완료 / 누적 6건

[7/250] 매물ID=41500585
[OK] 41500585 완료 / 누적 7건

[8/250] 매물ID=41563642
[OK] 41563642 완료 / 누적 8건

[9/250] 매물ID=41558812
[OK] 41558812 완료 / 누적 9건

[10/250] 매물ID=41387516
[OK] 41387516 완료 / 누적 10건

[11/250] 매물ID=41586727
[OK] 41586727 완료 / 누적 11건

[12/250] 매물ID=41309623
[OK] 41309623 완료 / 누적 12건

[13/250] 매물ID=41541267
[OK] 41541267 완료 / 누적 13건

[14/250] 매물ID=41295637
[OK] 41295637 완료 / 누적 14건

[15/250] 매물ID=41047149
[OK] 41047149 완료 / 누적 15건

[16/250] 매물ID=41548126
[OK] 41548126 완료 / 누적 16건

[17/250] 매물ID=41473416
[OK] 41473416 완료 /